# BamiBERT development experiment

Fit a shared span and relation encoder, then a student applicability cross-encoder. Use the reviewed SmartNotice v3.3 Drive release and its frozen development split. Evaluation uses reviewed draft extraction labels and rule-derived silver applicability labels. The reserved ordinary and web test sets stay outside this experiment.

Run from the repository root with SmartNotice, PyTorch, Transformers, and scikit-learn installed. LoRA also requires PEFT. Set `SMARTNOTICE_SPLIT_ROOT` to a private directory with train and validation notices and annotations, `assignments.jsonl`, `record.json`, and `checksums.json`. Set `SMARTNOTICE_BAMIBERT_DIR` to a local copy of the pinned checkpoint for an offline run. Set `SMARTNOTICE_RUN_DIR` to a fresh private output directory. For hosted use, install the SmartNotice wheel and mount an explicitly authorized copy of the same split.

Set `SMARTNOTICE_TUNING` to `lora` or `full` to choose the tuning mode. By default the notebook uses LoRA below 7 GiB of GPU memory. The local [notebook runner](../../../tools/run_bamibert_notebook.py) configures the input and output directories and saves the executed notebook beside the aggregate metrics.

See the [implementation plan](../../../docs/bamibert-implementation-2026-11-22.md) and [BamiBERT paper](https://arxiv.org/abs/2607.02259).


In [ ]:
import gc
import hashlib
import json
import os
import platform
import shutil
from dataclasses import asdict
from importlib.metadata import version
from importlib.resources import files
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from smartnotice.modeling.applicability import train_applicability
from smartnotice.modeling.data import choose_width, load_development, sha256, tokenize
from smartnotice.modeling.experiment import (
    Settings,
    evaluate_extraction,
    load_extractor,
    train_extraction,
    write_summary,
)
from transformers import AutoTokenizer

MODEL = "Qualcomm-AI-Research/BamiBERT"
REVISION = "57bc1340debbe4e348ec549047a763caebe4a977"
MODEL_SHA256 = "c8c8abe5914f2543bd6a65019b685dacf25ad39da5f01b680b082876e1947627"
root = Path(os.environ["SMARTNOTICE_SPLIT_ROOT"])
output = Path(os.environ.get("SMARTNOTICE_RUN_DIR", "reports/generated/bamibert/v3_3"))
output.mkdir(parents=True, exist_ok=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vram = torch.cuda.get_device_properties(device).total_memory if device.type == "cuda" else 0
settings = Settings(
    tuning=os.environ.get("SMARTNOTICE_TUNING", "lora" if vram < 7 * 2**30 else "full"),
    suppress_same_type_overlap=os.environ.get("SMARTNOTICE_SPAN_NMS") == "1",
)
print(device, flush=True)
checkpoint = os.environ.get("SMARTNOTICE_BAMIBERT_DIR") or snapshot_download(
    MODEL,
    revision=REVISION,
    allow_patterns=["*.json", "*.safetensors"],
)
checkpoint_path = Path(checkpoint)
weights_sha256 = sha256(checkpoint_path / "model.safetensors")
if weights_sha256 != MODEL_SHA256:
    raise ValueError("BamiBERT weights do not match the pinned release")
tokenizer = AutoTokenizer.from_pretrained(checkpoint, use_fast=True)
assert tokenizer.is_fast
print(tokenizer.__class__.__name__)

In [ ]:
documents, provenance = load_development(root)
width = choose_width(documents["train"], tokenizer, settings.length, settings.overlap)
tokenized = {
    split: [tokenize(doc, tokenizer, settings.length, settings.overlap, width) for doc in items]
    for split, items in documents.items()
}
run = {
    "model": MODEL,
    "revision": REVISION,
    "settings": asdict(settings),
    "model_weights_sha256": weights_sha256,
    "tokenizer_sha256": sha256(checkpoint_path / "tokenizer.json"),
    "tokenizer_class": tokenizer.__class__.__name__,
    "data": provenance,
    "max_width_subwords": width,
    "python": platform.python_version(),
    "platform": platform.platform(),
    "device": str(device),
    "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
    "versions": {name: version(name) for name in ("torch", "transformers", "scikit-learn")},
    "peft_version": version("peft") if settings.tuning == "lora" else None,
    "code_sha256": {
        p.name: hashlib.sha256(p.read_bytes()).hexdigest()
        for p in files("smartnotice.modeling").iterdir()
        if p.name.endswith(".py")
    },
    "label_scope": "development only",
    "counts": {
        split: {
            "documents": len(items),
            "windows": sum(len(d.windows) for d in items),
            "groups": len({d.document.group_id for d in items}),
        }
        for split, items in tokenized.items()
    },
}
write_summary(output / "run.json", run)
print(run["counts"], "max width:", width)

In [ ]:
resume = os.environ.get("SMARTNOTICE_RESUME_EXTRACTION")
if resume:
    prior = Path(resume)
    prior_run = json.loads((prior / "run.json").read_text(encoding="utf-8"))
    for key in ("model_weights_sha256", "tokenizer_sha256", "data", "max_width_subwords"):
        if prior_run[key] != run[key]:
            raise ValueError(f"Extraction run input differs: {key}")
    if Settings(**prior_run["settings"]) != settings:
        raise ValueError("Extraction run settings differ")
    saved = torch.load(prior / "extraction.pt", map_location="cpu")
    model = load_extractor(checkpoint, saved, tokenizer.pad_token_id, settings, device)
    extraction, predictions = evaluate_extraction(model, tokenized["validation"], settings)
    del model
    shutil.copyfile(prior / "extraction.pt", output / "extraction.pt")
    run["resumed_extraction"] = {
        "checkpoint_sha256": sha256(output / "extraction.pt"),
        "training_epoch_selected": saved["epoch"],
        "training_run_sha256": sha256(prior / "run.json"),
    }
    write_summary(output / "run.json", run)
    write_summary(output / "extraction_metrics.json", extraction)
else:
    extraction, predictions = train_extraction(
        tokenized["train"],
        tokenized["validation"],
        checkpoint,
        tokenizer.pad_token_id,
        width,
        settings,
        output,
        device,
    )
for key in ("span", "link_annotated_endpoints", "link_predicted_endpoints", "exact_event"):
    print(key, extraction[key].get("micro", extraction[key]))
gc.collect()
if device.type == "cuda":
    torch.cuda.empty_cache()

In [ ]:
applicability = train_applicability(
    documents["train"],
    documents["validation"],
    predictions,
    checkpoint,
    tokenizer,
    settings,
    output,
    device,
)
write_summary(
    output / "metrics.json",
    {
        "run": run,
        "extraction": extraction,
        "applicability": applicability,
    },
)
print("Annotated-event applicability Macro-F1:", applicability["annotated_events"]["macro_f1"])
print(
    "Matched predicted-event applicability:",
    applicability["predicted_events"]["matched_action_applicability"]["macro_f1"],
)